# Notebook 06: Full RLHF Pipeline -- Putting It All Together

**Sprint 1 of the RLHF Toolkit** | Frontier AI Lab Interview Preparation

---

This notebook wires together all three phases of RLHF into a single end-to-end pipeline:

**Pretrained LM --> [SFT] --> SFT Model --> [RM Training] --> Reward Model --> [PPO] --> Aligned Model**

We train on the Anthropic HH-RLHF dataset using GPT-2 (small enough for any GPU).

**Prerequisites**: Notebooks 01-05

**Runtime**: GPU required (Colab T4, RunPod, etc.). Full run ~45 min with GPT-2.

---
## 1. Self-Quiz (Active Recall)

Before reading any code, answer from memory:

1. **What are the 3 phases of RLHF? What is the input/output of each phase?**
2. **What data does each phase require?**
3. **How many models are in memory during PPO training? What does each one do?**
4. **What are the key hyperparameters for each phase?**
5. **Name 3 things that can go wrong in the RLHF pipeline and how you would diagnose them.**
6. **Could you skip any of the 3 phases? What would happen?**

<details>
<summary>Click to reveal answers after attempting</summary>

1. Phase 1: SFT (Supervised Fine-Tuning) -- takes pretrained LM + demonstration data, produces SFT model that can follow instructions. Phase 2: Reward Modeling -- takes SFT model + comparison/ranking data, produces reward model that scores responses. Phase 3: PPO -- takes SFT model + reward model + prompts, produces aligned model.

2. SFT: (prompt, good_response) pairs. RM: (prompt, response_A, response_B, preference) tuples. PPO: prompts only (reward comes from RM).

3. Four models: (1) Policy model (being optimized), (2) Reference model (frozen copy of SFT for KL), (3) Reward model (scoring responses), (4) Value model (estimating V(s) for GAE). In practice, policy and value share the backbone.

4. SFT: learning rate, epochs, data quality. RM: learning rate, margin in loss, data balance. PPO: KL coefficient (beta), clip range, learning rate, batch size, PPO epochs, generation temperature.

5. (a) Reward hacking -- RM score increases but quality degrades; diagnose by checking KL and sampling outputs. (b) Mode collapse -- all responses become similar; check entropy and diversity metrics. (c) KL explosion -- policy diverges too far; check KL curve, may need higher beta or adaptive controller.

6. Skip SFT: DeepSeek-R1-Zero showed this works but gets weird formatting/language mixing. Skip RM: use DPO instead (direct preference optimization) -- Sprint 2. Skip PPO: just do SFT, which is the most common approach (most deployed models). Can also do rejection sampling instead of PPO.

</details>

---
## 2. Setup

In [ ]:
!pip install -q torch transformers datasets trl accelerate matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any
from collections import defaultdict
import random
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
random.seed(42)
np.random.seed(42)

---
## 3. Pipeline Overview

```
                          RLHF Pipeline
                          =============

  [Phase 1: SFT]              [Phase 2: RM]              [Phase 3: PPO]
  ===============              =============              ==============
                                                          
  Pretrained LM               SFT Model                  SFT Model (policy)
       |                          |                       SFT Model (ref, frozen)
       | + demonstration          | + comparison          Reward Model (frozen)
       |   data                   |   data                Value Head
       v                          v                            |
  Supervised                 Train reward                     | + prompts
  fine-tuning                model (classify                  |   (no labels!)
       |                     preferred vs                     v
       v                     rejected)                   PPO training loop:
  SFT Model --------+            |                       generate -> score ->
                     |            v                       advantage -> update
                     |       Reward Model --------+            |
                     |                            |            v
                     +----------------------------+-->  Aligned Model
```

### What each phase does and why it's needed:

| Phase | Purpose | Data | Why needed |
|-------|---------|------|------------|
| **SFT** | Teach model to follow instructions | (prompt, response) pairs | Base LM just does next-token prediction; SFT teaches it the format of helpful responses |
| **RM** | Learn what humans prefer | (prompt, chosen, rejected) | Humans can't give real-time feedback during PPO; RM is a proxy for human preferences |
| **PPO** | Optimize model to generate preferred outputs | Prompts only | SFT imitates; PPO *optimizes*. The model explores and finds responses better than the demonstrations |

### Data requirements:
- **SFT**: ~10K-100K high-quality (prompt, response) pairs
- **RM**: ~50K-500K preference comparisons
- **PPO**: ~10K-100K diverse prompts (responses are generated, not provided)

---
## 4. Loading/Training Components

We'll train small versions of each component on the Anthropic HH-RLHF dataset.

Using GPT-2 (124M params) so this runs on any GPU.

In [ ]:
# ================================================================
# MODEL AND DATA SETUP
# ================================================================

MODEL_NAME = "gpt2"  # 124M params -- fits easily on T4/RunPod
MAX_LEN = 256         # Max sequence length (keep short for speed)
MAX_NEW_TOKENS = 64   # Max generation length

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Important for generation with padding

print(f"Model: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: {MAX_LEN}")

In [ ]:
# ================================================================
# LOAD AND PREPARE HH-RLHF DATA
# ================================================================

print("Loading Anthropic HH-RLHF dataset...")
hh_dataset = load_dataset("Anthropic/hh-rlhf", split="train")
print(f"Total examples: {len(hh_dataset)}")

# Sample subsets for each phase (keep small for demo)
hh_subset = hh_dataset.shuffle(seed=42).select(range(min(5000, len(hh_dataset))))

def extract_prompt_and_response(text):
    """Extract the last human turn as prompt and assistant response."""
    # HH-RLHF format: \n\nHuman: ... \n\nAssistant: ...
    parts = text.split("\n\nAssistant:")
    if len(parts) < 2:
        return None, None
    prompt = parts[0] + "\n\nAssistant:"
    response = parts[-1].strip()
    # Truncate for efficiency
    prompt = prompt[-300:]  # Keep last 300 chars of prompt
    response = response[:200]  # Keep first 200 chars of response
    return prompt, response

# Prepare data for each phase
sft_data = []       # (prompt, response) for SFT
rm_data = []        # (prompt, chosen, rejected) for RM
ppo_prompts = []    # prompts only for PPO

for example in hh_subset:
    chosen_prompt, chosen_response = extract_prompt_and_response(example['chosen'])
    rejected_prompt, rejected_response = extract_prompt_and_response(example['rejected'])
    
    if chosen_prompt and chosen_response and rejected_response:
        sft_data.append({'prompt': chosen_prompt, 'response': chosen_response})
        rm_data.append({
            'prompt': chosen_prompt,
            'chosen': chosen_response,
            'rejected': rejected_response
        })
        ppo_prompts.append(chosen_prompt)

# Split data across phases
n = len(sft_data)
sft_train = sft_data[:int(0.4*n)]         # 40% for SFT
rm_train = rm_data[int(0.4*n):int(0.8*n)] # 40% for RM
ppo_train_prompts = ppo_prompts[int(0.8*n):]  # 20% prompts for PPO

print(f"\nData splits:")
print(f"  SFT training: {len(sft_train)} examples")
print(f"  RM training:  {len(rm_train)} examples")
print(f"  PPO prompts:  {len(ppo_train_prompts)} prompts")

# Show an example
print(f"\n--- Example ---")
print(f"Prompt: {sft_data[0]['prompt'][:150]}...")
print(f"Chosen: {rm_data[0]['chosen'][:100]}...")
print(f"Rejected: {rm_data[0]['rejected'][:100]}...")

In [ ]:
# ================================================================
# PHASE 1: SUPERVISED FINE-TUNING (SFT)
# ================================================================

print("=" * 70)
print("PHASE 1: Supervised Fine-Tuning")
print("=" * 70)

class SFTDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        prompt_text = item['prompt'] + " "
        response_text = item['response'] + self.tokenizer.eos_token
        full_text = prompt_text + response_text
        
        encoding = self.tokenizer(
            full_text, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        
        # BUG FIX: Mask BOTH padding AND prompt tokens with -100.
        # As taught in Notebook 02, we only compute loss on response tokens.
        # Tokenize prompt alone to find the boundary.
        prompt_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
        prompt_len = min(len(prompt_ids), self.max_len)
        
        labels = input_ids.clone()
        # NOTE: tokenizer.padding_side="left" -- pads come BEFORE the prompt, so
        # masking labels[:prompt_len] from index 0 (a right-padding assumption)
        # would leave real prompt tokens unmasked. Offset by the pad length.
        pad_len = int((attention_mask == 0).sum().item())  # left-pad offset
        labels[: pad_len + prompt_len] = -100  # Mask padding + prompt tokens
        labels[attention_mask == 0] = -100     # Mask pad positions (either padding side)
        
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

# Create dataset and loader
sft_dataset = SFTDataset(sft_train[:500], tokenizer, MAX_LEN)  # Use 500 for speed
sft_loader = DataLoader(sft_dataset, batch_size=4, shuffle=True)

# Initialize SFT model
sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
sft_optimizer = AdamW(sft_model.parameters(), lr=5e-5, weight_decay=0.01)

# SFT training loop
SFT_EPOCHS = 2
sft_losses = []

print(f"Training SFT model for {SFT_EPOCHS} epochs on {len(sft_dataset)} examples...")
sft_model.train()

for epoch in range(SFT_EPOCHS):
    epoch_loss = 0
    n_batches = 0
    for batch in sft_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = sft_model(**batch)
        loss = outputs.loss
        
        sft_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
        sft_optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        sft_losses.append(loss.item())
    
    print(f"  Epoch {epoch+1}/{SFT_EPOCHS} -- Loss: {epoch_loss/n_batches:.4f}")

print("SFT training complete!")

In [ ]:
# ================================================================
# PHASE 2: REWARD MODEL TRAINING
# ================================================================

print("\n" + "=" * 70)
print("PHASE 2: Reward Model Training")
print("=" * 70)

class RewardModel(nn.Module):
    """Reward model: maps (prompt, response) to a scalar score.
    
    Architecture: GPT-2 backbone + linear head on last token's hidden state.
    """
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(model_name)
        self.config = self.backbone.config
        self.reward_head = nn.Linear(self.config.n_embd, 1)
        nn.init.zeros_(self.reward_head.bias)
        nn.init.normal_(self.reward_head.weight, std=1/(self.config.n_embd + 1)**0.5)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.backbone(
            input_ids, attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden = outputs.hidden_states[-1]  # (batch, seq, hidden)
        
        # Use last non-padding token's hidden state
        if attention_mask is not None:
            # NOTE: `attention_mask.sum(1) - 1` assumes RIGHT padding, but this
            # pipeline left-pads. cumsum().argmax() gives the index of the last
            # non-pad token regardless of padding side (cumsum first peaks there).
            last_idx = attention_mask.cumsum(dim=1).argmax(dim=1)  # (batch,)
            last_hidden = hidden[torch.arange(hidden.size(0)), last_idx]  # (batch, hidden)
        else:
            last_hidden = hidden[:, -1, :]
        
        reward = self.reward_head(last_hidden).squeeze(-1)  # (batch,)
        return reward

class RMDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        chosen_text = item['prompt'] + " " + item['chosen']
        rejected_text = item['prompt'] + " " + item['rejected']
        
        chosen_enc = self.tokenizer(
            chosen_text, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        rejected_enc = self.tokenizer(
            rejected_text, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        
        return {
            'chosen_ids': chosen_enc['input_ids'].squeeze(),
            'chosen_mask': chosen_enc['attention_mask'].squeeze(),
            'rejected_ids': rejected_enc['input_ids'].squeeze(),
            'rejected_mask': rejected_enc['attention_mask'].squeeze(),
        }

# Initialize reward model from SFT model weights (common practice)
reward_model = RewardModel(MODEL_NAME).to(device)
# Load SFT weights into backbone for better initialization
reward_model.backbone.load_state_dict(sft_model.state_dict())

rm_dataset = RMDataset(rm_train[:500], tokenizer, MAX_LEN)  # 500 for speed
rm_loader = DataLoader(rm_dataset, batch_size=4, shuffle=True)
rm_optimizer = AdamW(reward_model.parameters(), lr=1e-5, weight_decay=0.01)

# RM training loop
RM_EPOCHS = 2
rm_losses = []
rm_accuracies = []

print(f"Training reward model for {RM_EPOCHS} epochs on {len(rm_dataset)} pairs...")
reward_model.train()

for epoch in range(RM_EPOCHS):
    epoch_loss = 0
    epoch_correct = 0
    epoch_total = 0
    n_batches = 0
    
    for batch in rm_loader:
        chosen_ids = batch['chosen_ids'].to(device)
        chosen_mask = batch['chosen_mask'].to(device)
        rejected_ids = batch['rejected_ids'].to(device)
        rejected_mask = batch['rejected_mask'].to(device)
        
        # Score both
        chosen_reward = reward_model(chosen_ids, chosen_mask)
        rejected_reward = reward_model(rejected_ids, rejected_mask)
        
        # Bradley-Terry loss: -log(sigma(r_chosen - r_rejected))
        loss = -F.logsigmoid(chosen_reward - rejected_reward).mean()
        
        rm_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        rm_optimizer.step()
        
        # Accuracy: how often does RM rank chosen > rejected?
        correct = (chosen_reward > rejected_reward).float().sum().item()
        epoch_correct += correct
        epoch_total += len(chosen_reward)
        epoch_loss += loss.item()
        n_batches += 1
        rm_losses.append(loss.item())
        rm_accuracies.append(correct / len(chosen_reward))
    
    acc = epoch_correct / epoch_total if epoch_total > 0 else 0
    print(f"  Epoch {epoch+1}/{RM_EPOCHS} -- Loss: {epoch_loss/n_batches:.4f}, Accuracy: {acc:.3f}")

reward_model.eval()
print("Reward model training complete!")

---
## 5. Wiring the Pipeline

Now we connect all components into a single `RLHFPipeline` class:
- **Policy model** (SFT model + value head) -- generates responses, gets updated by PPO
- **Reference model** (frozen copy of SFT model) -- provides KL anchor
- **Reward model** (frozen) -- scores responses
- **Generation -> Scoring -> Advantage -> Update** loop

In [ ]:
class ValueHead(nn.Module):
    """Value head for PPO's value function."""
    def __init__(self, hidden_size, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, 1)
        nn.init.zeros_(self.linear.bias)
        nn.init.normal_(self.linear.weight, std=1e-2)
    
    def forward(self, hidden_states):
        return self.dropout(self.linear(hidden_states)).squeeze(-1)


class PolicyWithValueHead(nn.Module):
    """LM policy + value head for PPO."""
    def __init__(self, base_model):
        super().__init__()
        self.lm = base_model
        self.config = base_model.config
        self.v_head = ValueHead(self.config.n_embd)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.lm(
            input_ids, attention_mask=attention_mask,
            output_hidden_states=True
        )
        logits = outputs.logits
        hidden = outputs.hidden_states[-1]
        values = self.v_head(hidden)
        return logits, values
    
    def generate(self, **kwargs):
        return self.lm.generate(**kwargs)

In [ ]:
@dataclass
class RLHFConfig:
    """Configuration for the full RLHF pipeline."""
    # PPO
    ppo_epochs: int = 2
    clip_range: float = 0.2
    vf_coef: float = 0.1
    entropy_coef: float = 0.01
    max_grad_norm: float = 0.5
    
    # KL
    kl_coef: float = 0.1
    adaptive_kl: bool = True
    target_kl: float = 6.0  # target for PER-SEQUENCE summed KL (Ziegler/TRL convention)
    
    # GAE
    gamma: float = 1.0
    lam: float = 0.95
    
    # Generation
    max_new_tokens: int = 64
    temperature: float = 0.8
    top_p: float = 0.95
    
    # Training
    learning_rate: float = 1.41e-5
    batch_size: int = 4
    total_steps: int = 50


class RLHFPipeline:
    """Full RLHF pipeline: generation -> scoring -> advantage -> PPO update.
    
    This connects all three phases into a single training loop.
    """
    
    def __init__(
        self,
        config: RLHFConfig,
        policy: PolicyWithValueHead,
        ref_model: nn.Module,
        reward_model: nn.Module,
        tokenizer,
    ):
        self.config = config
        self.policy = policy
        self.ref_model = ref_model
        self.reward_model = reward_model
        self.tokenizer = tokenizer
        
        # Freeze ref and reward model
        for p in self.ref_model.parameters():
            p.requires_grad = False
        for p in self.reward_model.parameters():
            p.requires_grad = False
        
        self.optimizer = Adam(self.policy.parameters(), lr=config.learning_rate)
        
        # Adaptive KL controller
        self.kl_coef = config.kl_coef
        
        # Logging
        self.log = defaultdict(list)
    
    def _update_kl_coef(self, mean_kl: float):
        """Adaptive KL controller: adjust beta to maintain target KL.

        NOTE: the canonical ~6-nat target (Ziegler et al. 2019 / TRL) is a
        per-sequence SUMMED KL, so pass summed-per-sequence KL, not the
        per-token mean (unit mismatch otherwise).
        """
        if not self.config.adaptive_kl:
            return
        if mean_kl < self.config.target_kl / 1.5:
            self.kl_coef /= 1.5  # KL too low, reduce penalty
        elif mean_kl > self.config.target_kl * 1.5:
            self.kl_coef *= 1.5  # KL too high, increase penalty
        self.kl_coef = max(0.001, min(self.kl_coef, 10.0))  # Clamp
    
    @torch.no_grad()
    def _generate_and_score(self, prompts: List[str]) -> Dict[str, Any]:
        """Generate responses and score them with the reward model."""
        self.policy.eval()
        
        batch_data = []
        
        for prompt in prompts:
            # Encode prompt
            prompt_ids = self.tokenizer.encode(
                prompt, return_tensors="pt", truncation=True,
                max_length=MAX_LEN - self.config.max_new_tokens
            ).to(device)
            prompt_len = prompt_ids.shape[1]
            
            # Generate response
            gen_ids = self.policy.generate(
                input_ids=prompt_ids,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=True,
                temperature=self.config.temperature,
                top_p=self.config.top_p,
                pad_token_id=self.tokenizer.eos_token_id,
            )
            
            response_ids = gen_ids[0][prompt_len:]
            response_len = len(response_ids)
            
            if response_len == 0:
                continue
            
            # Get policy log probs and values
            logits, values = self.policy(gen_ids)
            log_probs = F.log_softmax(logits, dim=-1)
            resp_logprobs = log_probs[0, prompt_len-1:-1, :]
            token_logprobs = resp_logprobs.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)
            resp_values = values[0, prompt_len-1:-1]
            
            # Get reference log probs
            ref_out = self.ref_model(gen_ids)
            ref_log_probs = F.log_softmax(ref_out.logits, dim=-1)
            ref_resp_logprobs = ref_log_probs[0, prompt_len-1:-1, :]
            ref_token_logprobs = ref_resp_logprobs.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)
            
            # KL per token
            kl_per_token = token_logprobs - ref_token_logprobs
            
            # Reward model score (on full sequence)
            rm_score = self.reward_model(gen_ids, attention_mask=torch.ones_like(gen_ids))
            rm_score = rm_score.item()
            
            # Per-token rewards: KL penalty everywhere, RM reward at end
            per_token_rewards = -self.kl_coef * kl_per_token.clone()
            per_token_rewards[-1] += rm_score
            
            batch_data.append({
                'prompt': prompt,
                'gen_ids': gen_ids[0],
                'response_ids': response_ids,
                'prompt_len': prompt_len,
                'old_logprobs': token_logprobs,
                'values': resp_values,
                'rewards': per_token_rewards,
                'kl_per_token': kl_per_token,
                'rm_score': rm_score,
            })
        
        return batch_data
    
    @torch.no_grad()
    def _compute_advantages(self, batch_data: List[Dict]) -> List[Dict]:
        """GAE advantage estimation."""
        for item in batch_data:
            rewards = item['rewards']
            values = item['values']
            T = len(rewards)
            advantages = torch.zeros(T, device=device)
            last_gae = 0
            
            for t in reversed(range(T)):
                next_value = 0 if t == T - 1 else values[t + 1]
                delta = rewards[t] + self.config.gamma * next_value - values[t]
                advantages[t] = last_gae = delta + self.config.gamma * self.config.lam * last_gae
            
            item['advantages'] = advantages
            item['returns'] = advantages + values
        
        return batch_data
    
    def _ppo_update(self, batch_data: List[Dict]) -> Dict[str, float]:
        """PPO clipped surrogate update."""
        self.policy.train()
        
        total_pg = 0
        total_vf = 0
        total_ent = 0
        total_clip = 0
        n = 0
        
        for epoch in range(self.config.ppo_epochs):
            for item in batch_data:
                gen_ids = item['gen_ids'].unsqueeze(0)
                prompt_len = item['prompt_len']
                resp_ids = item['response_ids']
                old_lp = item['old_logprobs']
                adv = item['advantages']
                ret = item['returns']
                old_val = item['values']
                
                if len(adv) > 1:
                    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
                
                logits, values = self.policy(gen_ids)
                log_probs = F.log_softmax(logits, dim=-1)
                curr_lp = log_probs[0, prompt_len-1:-1, :].gather(
                    -1, resp_ids.unsqueeze(-1)
                ).squeeze(-1)
                curr_val = values[0, prompt_len-1:-1]
                
                # Policy loss
                ratio = torch.exp(curr_lp - old_lp)
                pg1 = -adv * ratio
                pg2 = -adv * torch.clamp(ratio, 1 - self.config.clip_range, 1 + self.config.clip_range)
                pg_loss = torch.max(pg1, pg2).mean()
                
                # Value loss
                vf_loss = 0.5 * ((curr_val - ret) ** 2).mean()
                
                # Entropy
                probs = F.softmax(logits[0, prompt_len-1:-1, :], dim=-1)
                entropy = -(probs * (probs + 1e-10).log()).sum(-1).mean()
                
                loss = pg_loss + self.config.vf_coef * vf_loss - self.config.entropy_coef * entropy
                
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), self.config.max_grad_norm)
                self.optimizer.step()
                
                clip_frac = (torch.abs(ratio - 1.0) > self.config.clip_range).float().mean()
                total_pg += pg_loss.item()
                total_vf += vf_loss.item()
                total_ent += entropy.item()
                total_clip += clip_frac.item()
                n += 1
        
        n = max(n, 1)
        return {'pg_loss': total_pg/n, 'vf_loss': total_vf/n,
                'entropy': total_ent/n, 'clip_frac': total_clip/n}
    
    def run_episode(self, prompts: List[str]) -> Dict[str, float]:
        """One full RLHF episode: generate -> score -> advantage -> update."""
        # Step 1+2: Generate and score
        batch_data = self._generate_and_score(prompts)
        
        if len(batch_data) == 0:
            return {'mean_reward': 0, 'mean_kl': 0, 'mean_len': 0}
        
        # Step 3: Compute advantages
        batch_data = self._compute_advantages(batch_data)
        
        # Step 4: PPO update
        update_stats = self._ppo_update(batch_data)
        
        # Aggregate stats
        mean_reward = np.mean([d['rm_score'] for d in batch_data])
        mean_kl = np.mean([d['kl_per_token'].mean().item() for d in batch_data])
        mean_len = np.mean([len(d['response_ids']) for d in batch_data])
        
        # Adaptive KL -- compare per-sequence SUMMED KL against target_kl=6.0;
        # mean_kl above is per-token and would mismatch the target's units.
        mean_seq_kl = np.mean([d['kl_per_token'].sum().item() for d in batch_data])
        self._update_kl_coef(mean_seq_kl)
        
        stats = {
            'mean_reward': mean_reward,
            'mean_kl': mean_kl,
            'mean_len': mean_len,
            'kl_coef': self.kl_coef,
            **update_stats,
        }
        
        for k, v in stats.items():
            self.log[k].append(v)
        
        return stats
    
    def train(self, prompts: List[str], total_steps: int = None):
        """Full training loop."""
        total_steps = total_steps or self.config.total_steps
        bs = self.config.batch_size
        
        print(f"Starting RLHF training for {total_steps} steps...")
        print(f"Batch size: {bs}, KL coef: {self.kl_coef:.4f}")
        print("=" * 80)
        
        for step in range(total_steps):
            batch_prompts = random.sample(prompts, min(bs, len(prompts)))
            stats = self.run_episode(batch_prompts)
            
            if step % 5 == 0 or step == total_steps - 1:
                print(
                    f"Step {step:3d}/{total_steps} | "
                    f"Reward: {stats['mean_reward']:+.3f} | "
                    f"KL: {stats['mean_kl']:.4f} | "
                    f"Len: {stats['mean_len']:.1f} | "
                    f"PG: {stats.get('pg_loss', 0):.4f} | "
                    f"VF: {stats.get('vf_loss', 0):.4f} | "
                    f"KL_coef: {stats.get('kl_coef', self.kl_coef):.4f}"
                )
        
        print("\nRLHF training complete!")

---
## 6. Training Run

In [ ]:
# ================================================================
# PHASE 3: PPO TRAINING (RLHF)
# ================================================================

print("\n" + "=" * 70)
print("PHASE 3: PPO Training (RLHF)")
print("=" * 70)

# Set up the 4 models:
# 1. Policy = SFT model + value head
policy = PolicyWithValueHead(deepcopy(sft_model)).to(device)

# 2. Reference = frozen copy of SFT model
ref_model_frozen = deepcopy(sft_model).to(device)
ref_model_frozen.eval()

# 3. Reward model = already trained above
reward_model.eval()

# 4. Value model = part of policy (shared backbone)

# Memory check
if torch.cuda.is_available():
    mem_allocated = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"\nGPU Memory: {mem_allocated:.1f} GB / {mem_total:.1f} GB allocated")
    print(f"We have 4 models loaded: policy, reference, reward model, value head")

# Configure RLHF
rlhf_config = RLHFConfig(
    ppo_epochs=2,
    batch_size=4,
    max_new_tokens=64,
    kl_coef=0.1,
    adaptive_kl=True,
    target_kl=6.0,
    learning_rate=1.41e-5,
    total_steps=40,
    temperature=0.8,
)

# Create pipeline
pipeline = RLHFPipeline(
    config=rlhf_config,
    policy=policy,
    ref_model=ref_model_frozen,
    reward_model=reward_model,
    tokenizer=tokenizer,
)

In [ ]:
# Run the RLHF training loop
pipeline.train(ppo_train_prompts, total_steps=rlhf_config.total_steps)

---
## 7. Evaluation: Before vs After

The real test: compare responses from the base model, SFT model, and RLHF model on the same prompts.

In [ ]:
# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

def generate_response(model, prompt, max_new_tokens=80, is_policy=False):
    """Generate a response from any model."""
    input_ids = tokenizer.encode(
        prompt, return_tensors="pt", truncation=True,
        max_length=MAX_LEN - max_new_tokens
    ).to(device)
    prompt_len = input_ids.shape[1]
    
    with torch.no_grad():
        if is_policy:
            gen_ids = model.generate(
                input_ids=input_ids, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.7, top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
        else:
            gen_ids = model.generate(
                input_ids, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.7, top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
    
    response = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

# Evaluation prompts
eval_prompts = [
    "\n\nHuman: What is the best way to learn a new language?\n\nAssistant:",
    "\n\nHuman: How do I make a good first impression at a job interview?\n\nAssistant:",
    "\n\nHuman: Can you explain how solar panels work?\n\nAssistant:",
    "\n\nHuman: What are some healthy breakfast ideas?\n\nAssistant:",
    "\n\nHuman: How can I reduce stress in my daily life?\n\nAssistant:",
]

print("=" * 80)
print("QUALITATIVE COMPARISON: Base vs SFT vs RLHF")
print("=" * 80)

for i, prompt in enumerate(eval_prompts):
    # Extract just the question for display
    question = prompt.split("Human:")[-1].split("\n")[0].strip()
    
    base_resp = generate_response(base_model, prompt)
    sft_resp = generate_response(sft_model, prompt)
    rlhf_resp = generate_response(policy, prompt, is_policy=True)
    
    print(f"\n{'='*80}")
    print(f"Question {i+1}: {question}")
    print(f"{'='*80}")
    print(f"\n  BASE ({len(base_resp.split())} words):")
    print(f"    {base_resp[:200]}")
    print(f"\n  SFT ({len(sft_resp.split())} words):")
    print(f"    {sft_resp[:200]}")
    print(f"\n  RLHF ({len(rlhf_resp.split())} words):")
    print(f"    {rlhf_resp[:200]}")

In [ ]:
# Quantitative evaluation: reward model scores
print("\n" + "=" * 70)
print("QUANTITATIVE COMPARISON")
print("=" * 70)

def score_response(reward_model, prompt, response):
    """Score a prompt-response pair with the reward model."""
    text = prompt + " " + response
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        score = reward_model(enc['input_ids'], enc['attention_mask'])
    return score.item()

base_scores, sft_scores, rlhf_scores = [], [], []
base_lens, sft_lens, rlhf_lens = [], [], []

# Score on more prompts
scoring_prompts = ppo_train_prompts[:20]

print("Scoring responses...")
for prompt in scoring_prompts:
    base_resp = generate_response(base_model, prompt)
    sft_resp = generate_response(sft_model, prompt)
    rlhf_resp = generate_response(policy, prompt, is_policy=True)
    
    base_scores.append(score_response(reward_model, prompt, base_resp))
    sft_scores.append(score_response(reward_model, prompt, sft_resp))
    rlhf_scores.append(score_response(reward_model, prompt, rlhf_resp))
    
    base_lens.append(len(base_resp.split()))
    sft_lens.append(len(sft_resp.split()))
    rlhf_lens.append(len(rlhf_resp.split()))

print(f"\n{'Metric':<25} {'Base':>10} {'SFT':>10} {'RLHF':>10}")
print("-" * 55)
print(f"{'Mean RM Score':<25} {np.mean(base_scores):>10.3f} {np.mean(sft_scores):>10.3f} {np.mean(rlhf_scores):>10.3f}")
print(f"{'Std RM Score':<25} {np.std(base_scores):>10.3f} {np.std(sft_scores):>10.3f} {np.std(rlhf_scores):>10.3f}")
print(f"{'Mean Length (words)':<25} {np.mean(base_lens):>10.1f} {np.mean(sft_lens):>10.1f} {np.mean(rlhf_lens):>10.1f}")
print(f"{'Std Length (words)':<25} {np.std(base_lens):>10.1f} {np.std(sft_lens):>10.1f} {np.std(rlhf_lens):>10.1f}")

# Pairwise win rate (by RM score)
rlhf_vs_sft_wins = sum(1 for r, s in zip(rlhf_scores, sft_scores) if r > s)
rlhf_vs_base_wins = sum(1 for r, b in zip(rlhf_scores, base_scores) if r > b)
sft_vs_base_wins = sum(1 for s, b in zip(sft_scores, base_scores) if s > b)
n = len(scoring_prompts)

print(f"\nWin rates (by RM score):")
print(f"  RLHF vs SFT:  {rlhf_vs_sft_wins}/{n} ({100*rlhf_vs_sft_wins/n:.0f}%)")
print(f"  RLHF vs Base: {rlhf_vs_base_wins}/{n} ({100*rlhf_vs_base_wins/n:.0f}%)")
print(f"  SFT vs Base:  {sft_vs_base_wins}/{n} ({100*sft_vs_base_wins/n:.0f}%)")

---
## 8. Visualization Dashboard

In [ ]:
# ================================================================
# COMPREHENSIVE TRAINING VISUALIZATION
# ================================================================

fig = plt.figure(figsize=(18, 14))
fig.suptitle('RLHF Pipeline -- Training Dashboard', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(3, 3, hspace=0.35, wspace=0.3)

log = pipeline.log
steps = range(len(log['mean_reward']))

# --- Row 1: Core PPO metrics ---

# 1. Reward over training
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, log['mean_reward'], 'b-', linewidth=1.5)
ax1.set_title('Mean RM Reward', fontweight='bold')
ax1.set_xlabel('Step')
ax1.set_ylabel('Reward')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=np.mean(log['mean_reward'][:5]), color='gray', linestyle='--', alpha=0.5, label='Initial')
ax1.legend(fontsize=8)

# 2. KL divergence over training
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(steps, log['mean_kl'], 'r-', linewidth=1.5)
if 'target_kl' in dir(rlhf_config):
    ax2.axhline(y=rlhf_config.target_kl, color='gray', linestyle='--', alpha=0.5, label=f'Target={rlhf_config.target_kl}')
ax2.set_title('Mean KL Divergence', fontweight='bold')
ax2.set_xlabel('Step')
ax2.set_ylabel('KL (nats)')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=8)

# 3. Policy entropy
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(steps, log['entropy'], 'g-', linewidth=1.5)
ax3.set_title('Policy Entropy', fontweight='bold')
ax3.set_xlabel('Step')
ax3.set_ylabel('Entropy (nats)')
ax3.grid(True, alpha=0.3)

# --- Row 2: Training dynamics ---

# 4. Response length
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(steps, log['mean_len'], 'purple', linewidth=1.5)
ax4.set_title('Mean Response Length', fontweight='bold')
ax4.set_xlabel('Step')
ax4.set_ylabel('Tokens')
ax4.grid(True, alpha=0.3)

# 5. Policy gradient loss
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(steps, log['pg_loss'], 'orange', linewidth=1.5)
ax5.set_title('Policy Gradient Loss', fontweight='bold')
ax5.set_xlabel('Step')
ax5.grid(True, alpha=0.3)

# 6. Value function loss
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(steps, log['vf_loss'], 'teal', linewidth=1.5)
ax6.set_title('Value Function Loss', fontweight='bold')
ax6.set_xlabel('Step')
ax6.grid(True, alpha=0.3)

# --- Row 3: Evaluation comparisons ---

# 7. Reward distribution: base vs SFT vs RLHF
ax7 = fig.add_subplot(gs[2, 0])
positions = [1, 2, 3]
bp = ax7.boxplot([base_scores, sft_scores, rlhf_scores], positions=positions,
                  patch_artist=True, widths=0.6)
colors = ['#ff9999', '#99ccff', '#99ff99']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax7.set_xticklabels(['Base', 'SFT', 'RLHF'])
ax7.set_title('RM Score Distribution', fontweight='bold')
ax7.set_ylabel('RM Score')
ax7.grid(True, alpha=0.3, axis='y')

# 8. Response length distribution
ax8 = fig.add_subplot(gs[2, 1])
bp2 = ax8.boxplot([base_lens, sft_lens, rlhf_lens], positions=positions,
                   patch_artist=True, widths=0.6)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
ax8.set_xticklabels(['Base', 'SFT', 'RLHF'])
ax8.set_title('Response Length Distribution', fontweight='bold')
ax8.set_ylabel('Words')
ax8.grid(True, alpha=0.3, axis='y')

# 9. Adaptive KL coefficient
ax9 = fig.add_subplot(gs[2, 2])
if 'kl_coef' in log:
    ax9.plot(steps, log['kl_coef'], 'brown', linewidth=1.5)
    ax9.set_title('Adaptive KL Coefficient', fontweight='bold')
    ax9.set_xlabel('Step')
    ax9.set_ylabel('Beta')
    ax9.grid(True, alpha=0.3)
else:
    ax9.text(0.5, 0.5, 'KL coef fixed', ha='center', va='center', transform=ax9.transAxes)
    ax9.set_title('KL Coefficient')

plt.savefig('/tmp/rlhf_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved to /tmp/rlhf_dashboard.png")

In [ ]:
# Phase 1 and 2 training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Phase 1 (SFT) and Phase 2 (RM) Training Curves', fontsize=13)

# SFT loss
axes[0].plot(sft_losses, 'b-', alpha=0.7)
axes[0].set_title('SFT Loss')
axes[0].set_xlabel('Batch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)

# RM loss
axes[1].plot(rm_losses, 'r-', alpha=0.7)
axes[1].set_title('RM Loss')
axes[1].set_xlabel('Batch')
axes[1].set_ylabel('Bradley-Terry Loss')
axes[1].grid(True, alpha=0.3)

# RM accuracy
axes[2].plot(rm_accuracies, 'g-', alpha=0.7)
axes[2].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[2].set_title('RM Accuracy')
axes[2].set_xlabel('Batch')
axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Hyperparameter Sensitivity

Understanding which hyperparameters matter most and how to diagnose failures is crucial for interviews.

### The Three Most Important Hyperparameters

| Hyperparameter | What it controls | Too low | Too high | Good default |
|---|---|---|---|---|
| **KL coefficient** ($\beta$) | How much to stay near reference | Reward hacking, mode collapse | Model barely changes | 0.1 - 0.2 (or adaptive) |
| **Learning rate** | Step size for policy updates | Slow convergence | Instability, KL explosion | 1e-5 to 5e-5 |
| **Batch size** | Gradient estimate quality | High variance, unstable | Slow (but more stable) | 64 - 512 (as large as possible) |

### Common Failure Modes and Diagnosis

**1. Reward Hacking**
- *Symptom*: RM score increases but actual quality degrades
- *Training curves*: Reward goes up, KL diverges, entropy drops
- *Diagnosis*: Sample outputs -- are they actually good? Compare with human eval
- *Fix*: Increase KL coefficient, use adaptive KL, improve reward model

**2. Mode Collapse**
- *Symptom*: All responses become very similar (low diversity)
- *Training curves*: Entropy drops sharply, reward plateaus
- *Diagnosis*: Generate multiple responses to same prompt -- are they all the same?
- *Fix*: Increase entropy bonus, decrease KL coefficient slightly, increase temperature

**3. KL Explosion**
- *Symptom*: KL diverges quickly, model becomes incoherent
- *Training curves*: KL shoots up, then reward and everything else go haywire
- *Diagnosis*: Check KL plot -- is it monotonically increasing without bound?
- *Fix*: Lower learning rate, increase KL coefficient, use adaptive KL controller, check for bugs in log-prob computation

In [ ]:
# Demonstrate diagnosing failure modes from training curves

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('How to Diagnose RLHF Failures from Training Curves', fontsize=13)
t = np.linspace(0, 10, 100)

# 1. Healthy training
ax = axes[0]
ax.set_title('Healthy Training', fontweight='bold', color='green')
ax.plot(t, 0.3 * np.log(t + 1), 'b-', label='Reward', linewidth=2)
ax.plot(t, 2 + 0.5 * np.log(t + 1), 'r-', label='KL', linewidth=2)
ax.plot(t, 8 - 0.3 * np.log(t + 1), 'g-', label='Entropy', linewidth=2)
ax.legend(fontsize=8)
ax.set_xlabel('Step')
ax.grid(True, alpha=0.3)
ax.text(5, -0.3, 'Reward increases steadily\nKL bounded\nEntropy decreases slowly', fontsize=8, ha='center')

# 2. Reward hacking
ax = axes[1]
ax.set_title('Reward Hacking', fontweight='bold', color='red')
ax.plot(t, 0.2 * t, 'b-', label='Reward (RM)', linewidth=2)
ax.plot(t, 0.5 * t, 'r-', label='KL (diverges!)', linewidth=2)
ax.plot(t, 8 - 0.8 * t, 'g-', label='Entropy (collapses!)', linewidth=2)
ax.plot(t, 0.15 * t - 0.01 * t**2, 'b--', label='Actual quality', linewidth=2, alpha=0.7)
ax.legend(fontsize=7)
ax.set_xlabel('Step')
ax.grid(True, alpha=0.3)
ax.text(5, -1, 'RM score up but actual quality peaks then drops\nKL diverges, entropy collapses', fontsize=8, ha='center')

# 3. KL explosion
ax = axes[2]
ax.set_title('KL Explosion', fontweight='bold', color='red')
ax.plot(t, 0.1 * np.sin(t) + 0.05 * t, 'b-', label='Reward (unstable)', linewidth=2)
ax.plot(t, np.exp(0.3 * t), 'r-', label='KL (explodes!)', linewidth=2)
ax.plot(t, 8 * np.exp(-0.3 * t), 'g-', label='Entropy (dies)', linewidth=2)
ax.legend(fontsize=7)
ax.set_xlabel('Step')
ax.set_ylim(-2, 15)
ax.grid(True, alpha=0.3)
ax.text(5, -1.5, 'KL grows exponentially\nReward oscillates\nFix: lower LR or higher beta', fontsize=8, ha='center')

plt.tight_layout()
plt.show()

---
## 10. System Design Teaser

### "How would you scale this to a 70B model?"

This is a common interview question at frontier labs. Here are the key challenges and solutions:

#### Challenge 1: Memory (4 models in memory)
With GPT-2 (124M), we need ~2 GB for all 4 models. With Llama-70B, we'd need ~560 GB just for weights in fp16!

**Solutions:**
- **Parameter sharing**: Policy and value model share the transformer backbone (we did this already)
- **LoRA**: Train only low-rank adapters, not full model. Reduces trainable params by 100-1000x
- **Offloading**: Keep reference model on CPU, move to GPU only when needed for KL computation
- **Quantization**: Load reference and reward models in 4-bit (QLoRA). Only policy needs full precision
- **Model parallelism**: Shard models across GPUs (tensor parallelism, pipeline parallelism)

#### Challenge 2: Compute (generation is the bottleneck)
PPO requires generating full responses from the policy at each step. For a 70B model generating 512 tokens, this is slow.

**Solutions:**
- **vLLM/TGI**: Use optimized inference engines with continuous batching, PagedAttention
- **Speculative decoding**: Use a small draft model to speed up generation
- **Async generation**: Generate while updating (overlapping compute and generation)
- **Shorter generations**: Reduce max_new_tokens (quality trade-off)

#### Challenge 3: Distributed Training

**Architecture sketch for distributed RLHF:**

```
  Generation Workers (N GPUs)         Training Workers (M GPUs)
  =============================       ===========================
  [Policy copy (inference only)]      [Policy (training)]
  [Reference model (frozen)]          [Value head]
  [Reward model (frozen)]             [Optimizer states]
           |                                    |
           | prompts, responses,                | updated weights
           | rewards, log_probs                 |
           v                                    v
      Experience Buffer  <===================>  PPO Update
```

This is the architecture used by OpenRLHF, TRL with DeepSpeed, etc.

**Key insight**: Separate generation from training. Generation can run on many GPUs with tensor parallelism. Training uses ZeRO-3 or FSDP for memory efficiency.

**Insider Tip:** In a real frontier lab interview, you'll be asked to design the RLHF system, not just implement the algorithm. Key questions to have answers ready for: 'How do you handle the 4 models in memory?' (parameter sharing, offloading, LoRA for ref model delta), 'How do you parallelize generation?' (separate generation and training GPU pools, vLLM for serving), 'How do you detect reward hacking in production?' (track reward vs. KL Pareto frontier, human eval checkpoints, held-out RM ensembles). Also know the recent alternatives: DPO removes the need for a separate RM and PPO entirely, GRPO removes the value model, and iterative/online DPO variants get the best of both worlds. Being able to compare these tradeoffs fluently is what separates senior from junior candidates.

---
## 11. "Why Does This Work?" -- Critical Thinking Prompts

### Q: Why 3 phases instead of end-to-end?

Could we train a single model end-to-end from human feedback? In theory, yes. In practice:

1. **Data efficiency**: Each phase uses different data, allowing specialization. SFT data (demonstrations) is expensive but small. RM data (comparisons) is cheaper and larger. PPO needs only prompts.

2. **Stability**: Each phase builds on a solid foundation. SFT gives a good starting point for the RM (the RM needs to evaluate SFT-quality outputs). The RM gives a stable reward signal for PPO.

3. **Modularity**: You can improve each phase independently. Better SFT data? Just retrain Phase 1. Better preference data? Retrain Phase 2. New RL algorithm? Replace Phase 3.

4. **Debugging**: If the final model is bad, you can isolate which phase went wrong.

### Q: Could you skip SFT?

**Yes!** DeepSeek-R1-Zero (January 2025) applied RL directly to a base model without SFT. Results:
- The model learned to reason (chain-of-thought emerged naturally)
- BUT it had formatting issues, language mixing, and readability problems
- DeepSeek-R1 (with SFT first) was much better on all fronts

**Lesson**: SFT teaches the model the "format" of good responses. RL teaches the model to optimize within that format. Without SFT, RL has to discover both format and content, which is harder.

### Q: Could you skip the reward model and use DPO instead?

**Yes!** Direct Preference Optimization (DPO) bypasses the RM entirely:
- Instead of: train RM, then use RM to train policy with PPO
- DPO does: directly optimize the policy from preference data
- The DPO loss implicitly optimizes the same objective as RLHF

**Trade-offs:**
- DPO is simpler (no RM, no PPO, no 4 models in memory)
- DPO is more stable (no RL instabilities)
- But PPO can potentially achieve better performance (it can explore beyond the preference data)
- PPO allows iterative improvement (generate -> score -> update), DPO is one-shot

**This is Sprint 2 of our toolkit!**

---
## Interview Question Bank: Full RLHF Pipeline

*These are representative interview-style questions on the full RLHF pipeline. If you're interviewing for a senior or principal ML role focused on alignment or post-training, expect to walk through this end-to-end, typically as a 45-minute deep dive.*

---

### Question 1: "Walk me through the RLHF pipeline end-to-end -- every design decision."

**What we're testing:** Comprehensive understanding of a complex ML system. This question reveals whether you've actually built or deeply studied these systems vs. just read about them.

**Good answer:** Describes the 3 phases correctly:
1. **Phase 1 (SFT):** Fine-tune a pretrained LM on high-quality instruction-response data to teach it the response format and basic helpfulness.
2. **Phase 2 (Reward Model):** Collect human preference data (which of two responses is better?), train a reward model using Bradley-Terry loss.
3. **Phase 3 (PPO):** Use the reward model as a signal to optimize the SFT model via PPO, with a KL penalty against the SFT model as reference.

Mentions key hyperparameters for each phase and knows that the RM and policy are both derived from the SFT model.

**Great answer (Principal-level):** Covers all of the above with depth on design decisions at each stage:

**Phase 1 decisions:** How much SFT data (100K-1M)? How many epochs (2-4)? Learning rate schedule? What to mask in the loss? How to evaluate SFT quality before proceeding to Phase 2? The quality of SFT directly bounds the quality of RLHF -- a bad SFT model produces poor reference policy and bad initial responses for preference collection.

**Phase 2 decisions:** RM architecture (same size as policy? smaller for efficiency?), how many preference pairs (500K-2M), annotation quality control (kappa > 0.7), handling ties and ambiguous preferences, evaluation beyond accuracy (calibration, robustness, adversarial probing), how to detect and mitigate length bias.

**Phase 3 decisions:** KL coefficient (start with $\beta=0.1$, use adaptive scheduling), PPO hyperparameters (clip epsilon 0.2, 1-4 epochs per batch), generation strategy (batch size, max length, temperature), when to stop training (reward plateau + human eval confirmation), and critically: how to monitor for reward hacking.

**The interaction between phases:** SFT quality affects preference data distribution which affects RM quality which affects PPO stability. Iterative RLHF: after Phase 3, go back to Phase 2 with new preference data on the improved policy, retrain RM, run Phase 3 again.

**Red flag:** Can't describe all 3 phases. Doesn't mention the KL penalty. Treats each phase as independent (doesn't understand the interactions). Doesn't know about reward hacking.

**Follow-up (the diagnostic question):** "You notice reward going up but human evaluators say quality is getting worse. Diagnose." This is the most important follow-up. The answer: This is reward hacking / overoptimization. The policy has found features of the response that the RM scores highly but that don't correspond to actual quality. Steps: (1) Compare RM score distribution with human rankings on a fresh sample, (2) Check if responses are getting longer (length hacking), more verbose, or repetitive, (3) Inspect high-reward responses manually -- look for patterns the RM might be exploiting, (4) Check KL divergence -- if it's very high, the policy has moved far from the reference and may be in an exploitable region. Mitigation: increase KL coefficient, retrain RM on new policy's outputs, use an ensemble of RMs, add length penalties, or cap training early.

---

### Question 2: "Compare RLHF vs DPO -- when would you use each?"

**What we're testing:** Awareness of the modern landscape and practical engineering judgment.

**Good answer:** DPO (Direct Preference Optimization) eliminates the need for a separate reward model and PPO training. Instead, it directly optimizes the policy on preference data by treating the LM itself as an implicit reward model. The DPO loss is derived from the closed-form solution to the KL-regularized RLHF objective. RLHF is more complex but more flexible; DPO is simpler but limited to offline data.

**Great answer (Principal-level):** Discusses the full trade-off space:

**When to use DPO:** (1) Limited compute budget -- DPO is much cheaper (no generation loop, no RM training, no 4-model problem), (2) Strong existing preference data -- DPO works well when you have high-quality offline preference data, (3) Fast iteration -- DPO training is fast (similar cost to SFT), enabling rapid experimentation.

**When to use RLHF (PPO):** (1) Maximum quality -- PPO with a good RM consistently outperforms DPO at the frontier, (2) Online exploration -- PPO generates new responses and learns from them (online RL), while DPO only learns from the static dataset (offline RL), (3) Reward model reuse -- a trained RM can be used for best-of-N sampling, monitoring, and evaluation beyond just training, (4) Iterative improvement -- PPO naturally supports iterating with fresh data.

**The middle ground:** Iterative DPO / online DPO -- generate new responses with the current policy, collect preferences on them, retrain with DPO. This bridges the online/offline gap. Also: RLHF for the first training run (expensive but high quality), then DPO for subsequent iterations (cheaper, using the RLHF model's outputs as the new reference).

**Emerging alternatives:** GRPO (Group Relative Policy Optimization), KTO (Kahneman-Tversky Optimization), IPO (Identity Preference Optimization). These all simplify different aspects of RLHF while trying to retain quality.

**Red flag:** Claims DPO is strictly better than RLHF (or vice versa). Doesn't understand the online vs offline distinction. Can't derive DPO from the KL-regularized RLHF objective.

**Follow-up:** "If you could only choose one method for a production system and had unlimited compute, which would you choose and why?" (This is an opinion question -- there's no single right answer, but the reasoning matters. Strong candidates say PPO because online learning from fresh data is fundamentally more powerful than offline optimization on static data, but acknowledge that the engineering complexity is significant and DPO may be the right practical choice for many teams.)

---
## Production Implementation Notes: Full RLHF Pipeline at Frontier Scale

*What the full RLHF pipeline looks like when you're training a model that will serve millions of users. This is the production reality behind the 3-phase diagram.*

### The Textbook vs. Reality Gap

| Component | Textbook Version (this notebook) | Production Version (Anthropic/OpenAI scale) |
|-----------|--------------------------------|-------------------------------------------|
| **Pipeline** | Sequential: SFT -> RM -> PPO, run once | Iterative: multiple rounds, retraining RM on new policy outputs |
| **SFT data** | 1K examples, GPT-2 | 100K-1M expert-curated examples, 70B+ model |
| **Preference data** | Anthropic HH-RLHF (public) | 500K-2M proprietary preference pairs, $5-30M in annotation cost |
| **RM evaluation** | Accuracy on held-out set | Accuracy + calibration + robustness + adversarial + per-category analysis |
| **PPO training** | Single GPU, minutes | 256-1024 GPUs, 1-2 weeks per iteration |
| **Stopping criterion** | Fixed number of steps | Human evaluation agreement + reward curves + safety checks |
| **Total pipeline** | ~1 hour on one GPU | 4-8 weeks across hundreds of GPUs, multiple iterations |

### Full Pipeline Scale Numbers

- **Total compute for one RLHF iteration (70B model):**
  - Phase 1 (SFT): 64-256 H100 GPUs, 4-12 hours, ~$50K-$200K
  - Phase 2 (RM): 64-128 H100 GPUs, 6-24 hours, ~$50K-$150K
  - Phase 3 (PPO): 256-1024 H100 GPUs, 1-2 weeks -- roughly $0.5M-$2M in GPU cost at ~$2-3/H100-hr (full program cost is higher with experiments and ablations)
  - Data annotation: $5M-$30M per round
  - Human evaluation: $50K-$200K per checkpoint
  - **Total per iteration: ~$6M-$35M, dominated by data annotation**

- **Iterations:** Frontier labs run 2-5 RLHF iterations before shipping. Each iteration uses preference data collected on the previous iteration's outputs.

- **The inner loop of PPO:** Generate batch (256-1024 responses) -> Score with RM -> Compute advantages -> PPO update (1-4 epochs) -> Repeat. This loop runs thousands of times per PPO training run.

### Engineering Challenges at Pipeline Scale

1. **Pipeline orchestration:** Coordinating SFT training, RM training, data annotation, PPO training, and evaluation across hundreds of GPUs and human annotators is a major ops challenge. Teams use custom MLOps platforms with job scheduling, artifact tracking, and automated evaluation.

2. **Data flywheel management:** Each RLHF iteration produces a better model, which generates better responses, which need new preference annotations, which train a better RM, which improves PPO. Managing this data flywheel -- version control, quality tracking, decontamination -- is a full-time job for multiple engineers.

3. **Safety during training:** RLHF can inadvertently train the model to be less safe (if the RM doesn't capture safety well) or excessively cautious (if safety is over-weighted). Continuous safety evaluation with red-teaming is essential throughout PPO training.

4. **Evaluation bottleneck:** The hardest part of the pipeline is knowing when to stop. Automated metrics (reward, KL, benchmark scores) are necessary but not sufficient. Human evaluation is the gold standard but expensive and slow. Teams typically evaluate 3-5 checkpoints per PPO run with 500-2000 human comparisons each.

5. **Reproducibility across iterations:** Comparing RLHF iteration N to iteration N-1 requires careful experimental controls. Changes in annotation guidelines, annotator pool, or evaluation prompts can confound the comparison.

### The Production Monitoring Dashboard

A real RLHF monitoring dashboard shows (all in real-time):

- **Reward curves:** Mean reward per batch, with confidence intervals. Should increase and plateau.
- **KL curves:** KL divergence from reference model. Should increase gradually. Target: 5-15 nats.
- **Reward vs KL scatter:** The key plot. Shows reward efficiency -- how much reward per unit of KL.
- **Sample quality panel:** Random samples from the current policy, side-by-side with SFT baseline.
- **Safety metrics:** Refusal rates, toxicity scores, rule violations -- tracked per batch.
- **Benchmark scores:** MMLU, HumanEval, GSM8K, etc. -- evaluated at regular checkpoint intervals.
- **Generation statistics:** Response length distribution, vocabulary diversity, repetition rates.
- **Infrastructure metrics:** GPU utilization, generation throughput, training throughput, communication overhead.

### When Things Go Wrong (Real Production Failure Modes)

- **Reward hacking detected:** Human eval shows quality degrading while RM scores increase. Response: stop training, roll back to last good checkpoint, increase KL coefficient, retrain RM with fresh preference data.
- **Mode collapse:** Model starts generating very similar responses regardless of prompt. Entropy has collapsed. Response: increase entropy bonus, reduce learning rate, check for data issues.
- **Safety regression:** Model becomes less safe during PPO (starts complying with harmful requests). Response: emergency stop, audit safety-specific reward signal, add safety-specific RM or constraint.
- **Capability loss:** Model gets more helpful but loses capabilities (worse at math or coding). Response: add capability-preservation objectives, check if RM is biased toward style over substance.

---
## How This Gets Tested in Interviews

### Where Full RLHF Pipeline Questions Appear

| Company | Round | Format | Duration | Depth |
|---------|-------|--------|----------|-------|
| **Anthropic** | Onsite (dedicated RLHF round) | Deep discussion + system design + diagnosis | 45-60 min | The deepest -- this is their core technology. Expect follow-ups on every phase |
| **OpenAI** | Onsite (research/engineering) | Discussion + whiteboard design | 45 min | Deep -- full pipeline walkthrough with emphasis on failure modes |
| **DeepMind** | Onsite (ML systems design) | System design | 45 min | Moderate to deep -- emphasis on scalability and evaluation |
| **Meta (GenAI)** | Onsite | System design + discussion | 30-45 min | Moderate -- practical pipeline design, less theoretical |
| **xAI / Mistral / Cohere** | Onsite or final round | Discussion | 30 min | Moderate -- focus on practical trade-offs and alternatives to RLHF |

### The Interview Flow (What to Expect)

A typical Anthropic/OpenAI RLHF interview follows this pattern:

1. **"Walk me through RLHF end-to-end."** (10-15 min) -- They want to see you can explain all 3 phases clearly and correctly. This is the warm-up.

2. **Deep dive into one phase** (15-20 min) -- They'll pick the phase where your answer was thinnest, or the one most relevant to the team. Common deep dives:
   - "Tell me about your reward modeling approach" (Anthropic loves this)
   - "How do you handle the PPO training loop at scale?" (OpenAI loves this)
   - "How do you evaluate the final model?" (Everyone asks this)

3. **Failure mode diagnosis** (10-15 min) -- "Something went wrong. Here are the symptoms. Diagnose." Common scenarios:
   - Reward increasing, quality decreasing (reward hacking)
   - KL exploding (policy divergence)
   - Model becoming sycophantic (RM captures agreeableness as quality)
   - Model becoming overly cautious (safety over-optimization)

4. **Alternatives and future directions** (5-10 min) -- "If you could redesign RLHF from scratch, what would you change?" or "When would you use DPO instead?"

### Senior vs. Principal Expectations

**senior ML engineer:**
- Walk through all 3 phases correctly with key hyperparameters
- Describe the interaction between phases (SFT quality affects RM which affects PPO)
- Know the major failure modes (reward hacking, mode collapse)
- Implement key components given guidance
- Compare RLHF and DPO at a high level

**principal ML engineer:**
- All of the above, plus:
- Design the full production pipeline: data annotation workflow, training infrastructure, evaluation system, monitoring dashboard
- Quantify costs and timelines: how many GPUs, how long, how much data, how many annotators
- Discuss iterative RLHF: when to retrain the RM, how to collect new preference data, how to measure improvement across iterations
- Have deep opinions on open research questions: is RL necessary? Is the reward model the bottleneck? Will RLHF be replaced?
- Propose novel improvements: you should have at least one concrete idea for making the pipeline better (not necessarily published, but well-reasoned)
- Discuss safety throughout: how safety is maintained at each phase, how RLHF interacts with Constitutional AI, RLAIF
- Know the full landscape: RLHF, DPO, KTO, GRPO, IPO, rejection sampling, best-of-N -- when to use each

### Preparation Checklist

- [ ] Draw the full RLHF pipeline from memory with all design decisions at each stage
- [ ] Practice the "45-minute walkthrough" -- time yourself explaining RLHF end-to-end with depth on each phase
- [ ] Have 5 failure mode scenarios ready: symptoms, diagnosis, and fix for each
- [ ] Know the numbers: GPU count, training time, data size, annotation cost, for each phase
- [ ] Be ready for the DPO comparison: derive DPO from the KL-regularized RLHF objective
- [ ] Have a "what I would change about RLHF" answer ready (shows research taste)
- [ ] Be ready for: "You have a 6-month timeline, a team of 10 engineers, and 1000 GPUs. Design the RLHF pipeline for a new model." (This is a full system design question -- scope, prioritize, discuss trade-offs)
- [ ] Prepare for Anthropic specifically: know Constitutional AI, RLAIF, how they differ from standard RLHF

---
## 12. Flashcard Summary

| # | Question | Answer |
|---|----------|--------|
| 1 | What are the 3 phases of RLHF? | Phase 1: SFT (supervised fine-tuning). Phase 2: Reward model training. Phase 3: PPO (RL optimization). |
| 2 | What data does each phase need? | SFT: (prompt, good_response). RM: (prompt, chosen, rejected). PPO: prompts only. |
| 3 | How many models are in memory during PPO? | Four: policy, reference (frozen SFT copy for KL), reward model (frozen), value head (usually shares backbone with policy). |
| 4 | What is the role of the reference model? | Provides a KL anchor. The KL penalty $\beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})$ prevents the policy from diverging too far from the SFT model. |
| 5 | What is an adaptive KL controller? | Dynamically adjusts $\beta$ to maintain a target KL: if KL too low, decrease $\beta$ (allow more exploration); if KL too high, increase $\beta$ (constrain more). |
| 6 | What is reward hacking? How to detect it? | The policy exploits RM weaknesses to get high scores on bad outputs. Detect: RM score increases but sampled outputs look worse. KL diverges, entropy drops. |
| 7 | What is mode collapse in RLHF? | The policy converges to producing a single type of response for all prompts. Detect: low entropy, all responses look similar. Fix: increase entropy bonus. |
| 8 | What is KL explosion? How to fix it? | KL grows exponentially, policy becomes incoherent. Fix: lower learning rate, higher $\beta$, use adaptive KL controller, check log-prob computation for bugs. |
| 9 | Why use Bradley-Terry loss for RM? | $\mathcal{L} = -\log \sigma(r_w - r_l)$ models pairwise preference probability. It's a proper scoring rule and matches how preference data is collected. |
| 10 | Why SFT before RM training? | The RM needs to evaluate SFT-quality outputs. If initialized from a base model, the RM would evaluate base model outputs, which are different from what PPO will generate. |
| 11 | What happens if you skip SFT? | DeepSeek-R1-Zero showed RL alone can work, but produces formatting issues, language mixing, and readability problems. SFT teaches the "format" of good responses. |
| 12 | What is the main alternative to PPO for RLHF? | DPO (Direct Preference Optimization): directly optimizes the policy from preference data without an explicit RM. Simpler, more stable, but potentially less powerful. |
| 13 | How to scale RLHF to 70B models? | Key: separate generation from training, use LoRA/QLoRA, offload ref model to CPU, quantize frozen models, use tensor/pipeline parallelism, optimized inference (vLLM). |
| 14 | What is the generation bottleneck in RLHF? | PPO requires generating full responses at each step. For large models, autoregressive generation is slow. Solutions: speculative decoding, async generation, shorter max length. |
| 15 | What is reward model overoptimization? | Gao et al. (2023) showed that as you optimize more against the RM, RM score increases but true quality peaks then decreases. The proxy (RM) diverges from the ground truth (human preferences). |

---
## 13. Paper Guide

### Primary Paper 1: "Secrets of RLHF in Large Language Models Part I: PPO"
**Zheng et al. (2023)** | https://arxiv.org/abs/2307.04964

**Why this paper matters:**
One of the few papers that details practical PPO engineering for LMs. Most RLHF papers gloss over implementation.

**Key takeaways for interviews:**
- Process reward models (per-step) vs outcome reward models (end-of-sequence) and their tradeoffs
- PPO-max variant with reward clipping and normalization for stability
- Reward model calibration and data balancing strategies
- The importance of advantage normalization (Section 3.2)

**Read**: Sections 3 (PPO details) and 4 (RM overoptimization) most carefully.

---

### Primary Paper 2: "Scaling Laws for Reward Model Overoptimization"
**Gao et al. (2023)** | https://arxiv.org/abs/2210.10760

**Why this paper matters:**
Quantifies the fundamental tension in RLHF: optimizing against a proxy (RM) eventually hurts true quality.

**Key takeaways for interviews:**
- **Gold reward** (true human preference) vs **proxy reward** (RM score) -- they diverge!
- Proxy reward increases monotonically with optimization, but gold reward follows an inverted U
- Scaling laws: larger RMs delay overoptimization but don't eliminate it
- BoN (Best-of-N) sampling is a simple baseline that's competitive with PPO
- The KL penalty mainly slows movement in KL -- the gold-reward-vs-KL frontier is largely unchanged; KL penalties help stability/off-policy reuse more than they fix Goodharting

**Read**: Figures 1-3 and the scaling law analysis (Section 4).

---

### Additional References:
- **Ouyang et al. (2022)**: InstructGPT -- the foundational RLHF paper from OpenAI
- **Bai et al. (2022)**: Anthropic's "Training a Helpful and Harmless Assistant" -- RLHF at scale
- **Touvron et al. (2023)**: Llama-2 -- rejection sampling + PPO combined
- **Rafailov et al. (2023)**: DPO -- the main alternative to PPO (Sprint 2)
- **DeepSeek-AI (2025)**: DeepSeek-R1 -- RL without SFT (R1-Zero) and with SFT (R1)

### Recent Papers (2024-2025) -- Know for Interviews:
- **Online DPO / Iterative DPO** (Xu et al. 2024, Xiong et al. 2024): Bridging offline DPO with online exploration -- combines simplicity of DPO with data freshness of PPO
- **REBEL** (Gao et al. 2024): REgression to RELative REward Based RL -- replaces PPO with regression on relative rewards; not a best-of-N method
- **Llama 3** (Meta 2024): The most detailed open account of a modern post-training pipeline at scale: SFT + rejection sampling + DPO -- explicitly no PPO (Llama 2 used PPO)
- **Constitutional AI** (Bai et al. 2022): Anthropic's approach to self-improvement through AI-generated feedback -- foundational for understanding RLAIF